# 05 — Augmentation Training: M1 (Blind Aug) and M2 (Label-Aware Aug)

Trains M1 and M2, then checks the mandatory gate before M3+ can proceed.

| Model | Augmentation | Label-aware? |
|---|---|---|
| M1 | Curriculum (p=optimal) | No — strategies applied without respecting occlusion_lvl |
| M2 | Curriculum (p=optimal) | Yes — budget table controls strategy per occlusion_lvl |

**Mandatory gate (Step 9):**
`M2_AP_hard - M1_AP_hard >= 1.0`
If this is NOT met → STOP and revise the augmentation curriculum before proceeding to M3+.

**Prerequisites:**
- Notebook `02_aug_sweep` must have completed (optimal p recorded)
- Notebook `04_depth_precomputation` must have passed (depth maps on disk)
- Update `aug_probability` in `configs/m1_blind_aug.yaml` and `configs/m2_label_aug.yaml` with the optimal p from sweep.

In [ ]:
import sys
sys.path.insert(0, '..')

from pathlib import Path
import pandas as pd
import torch
from torch.utils.data import DataLoader

from src.config import load_config, set_all_seeds
from src.datasets import KITTIDataset, collate_fn
from src.logger import ExperimentLogger
from src.metrics import compute_kitti_ap, sample_to_annotation
from src.plotting import create_narrative_template

ROOT = Path('..').resolve()
TABLES_DIR = ROOT / 'results' / 'tables'
TABLES_DIR.mkdir(parents=True, exist_ok=True)

RUN_MODE = 'local'   # change to 'kaggle' for full training
SMOKE_TEST = (RUN_MODE == 'local')
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Load optimal p from sweep (or set manually)
optimal_p_file = ROOT / 'results' / 'optimal_aug_p.txt'
if optimal_p_file.exists():
    OPTIMAL_P = float(optimal_p_file.read_text().strip())
    print(f'Optimal p loaded from file: {OPTIMAL_P}')
else:
    OPTIMAL_P = 0.4  # fallback if sweep not yet run
    print(f'[WARN] optimal_aug_p.txt not found. Using default p={OPTIMAL_P}')
    print('Run notebook 02_aug_sweep first and update aug_probability in M1/M2 configs.')

print(f'Device: {device}  |  Mode: {RUN_MODE}')

In [ ]:
def train_and_evaluate(config_path, model_id, run_mode, smoke_test):
    """Train a model variant and return val AP metrics."""
    from ultralytics import YOLO

    cfg = load_config(config_path, overrides={
        'run_mode': run_mode,
        'epochs': 3 if smoke_test else 100,
        'data_limit': 100 if smoke_test else None,
    })
    cfg.ensure_dirs()
    set_all_seeds(cfg.seed)

    val_ds = KITTIDataset(cfg.kitti_root, split='val', imgsz=cfg.imgsz)
    val_loader = DataLoader(
        val_ds, batch_size=cfg.batch, shuffle=False,
        collate_fn=collate_fn, num_workers=0
    )

    model = YOLO('yolov8s.pt')
    logger = ExperimentLogger(
        run_id=f'{model_id}_seed{cfg.seed}',
        log_dir=cfg.log_dir,
        config=vars(cfg),
    )

    with logger:
        model.train(
            data=str(ROOT / 'data' / 'kitti_ultralytics.yaml'),
            epochs=cfg.epochs,
            imgsz=cfg.imgsz,
            batch=cfg.batch,
            optimizer=cfg.optimizer,
            lr0=cfg.lr0,
            lrf=cfg.lrf,
            momentum=cfg.momentum,
            weight_decay=cfg.weight_decay,
            warmup_epochs=cfg.warmup_epochs,
            amp=cfg.amp,
            seed=cfg.seed,
            project=str(cfg.checkpoint_dir),
            name=model_id,
            exist_ok=True,
            device=device,
        )

        best_ckpt = cfg.checkpoint_dir / model_id / 'weights' / 'best.pt'
        best_model = YOLO(str(best_ckpt))
        best_model.model.eval()

        predictions, annotations = [], []
        with torch.no_grad():
            for batch in val_loader:
                imgs = batch['image'].to(device)
                results = best_model(imgs, verbose=False)
                for i, r in enumerate(results):
                    boxes = r.boxes.xyxyn.cpu().numpy() if r.boxes is not None and len(r.boxes) > 0 else []
                    scores = r.boxes.conf.cpu().numpy() if r.boxes is not None and len(r.boxes) > 0 else []
                    predictions.append({'image_id': batch['image_id'][i], 'boxes': boxes, 'scores': scores})
                    annotations.append(sample_to_annotation(batch, i))

        ap_easy = compute_kitti_ap(predictions, annotations, 'easy')
        ap_mod  = compute_kitti_ap(predictions, annotations, 'moderate')
        ap_hard = compute_kitti_ap(predictions, annotations, 'hard')
        logger.log_metrics({'AP_easy': ap_easy, 'AP_mod': ap_mod, 'AP_hard': ap_hard})

    create_narrative_template(model_id, cfg.narratives_dir)
    return ap_easy, ap_mod, ap_hard, cfg

In [ ]:
# ── Train M1: Blind Augmentation ──────────────────────────────────────────────
print('='*60)
print('Training M1: Blind Augmentation')
print('='*60)

m1_easy, m1_mod, m1_hard, m1_cfg = train_and_evaluate(
    ROOT / 'configs' / 'm1_blind_aug.yaml', 'M1', RUN_MODE, SMOKE_TEST
)
print(f'M1  AP_easy={m1_easy:.2f}  AP_mod={m1_mod:.2f}  AP_hard={m1_hard:.2f}')

In [ ]:
# ── Train M2: Label-Aware Augmentation ────────────────────────────────────────
print('='*60)
print('Training M2: Label-Aware Augmentation')
print('='*60)

m2_easy, m2_mod, m2_hard, m2_cfg = train_and_evaluate(
    ROOT / 'configs' / 'm2_label_aug.yaml', 'M2', RUN_MODE, SMOKE_TEST
)
print(f'M2  AP_easy={m2_easy:.2f}  AP_mod={m2_mod:.2f}  AP_hard={m2_hard:.2f}')

In [ ]:
# ── Mandatory Gate (Step 9) ───────────────────────────────────────────────────
delta = m2_hard - m1_hard
print('\n' + '='*60)
print('MANDATORY GATE CHECK')
print('='*60)
print(f'M1 AP_hard:  {m1_hard:.2f}')
print(f'M2 AP_hard:  {m2_hard:.2f}')
print(f'Delta:       {delta:+.2f}')
print(f'Required:    >= +1.0')

if delta >= 1.0:
    print('\n[PASS] Gate passed. Proceed to notebook 06_fusion.')
else:
    print('\n[FAIL] Gate NOT passed.')
    print('STOP: Do not proceed to M3+.')
    print('Action required: Revise the augmentation curriculum in src/augmentation.py')
    print('Likely issues:')
    print('  - Aug probability too low or too high')
    print('  - Budget table not properly restricting lvl=2 instances')
    print('  - Curriculum epoch schedule mismatch')
    raise AssertionError(
        f'Gate FAILED: M2_AP_hard - M1_AP_hard = {delta:.2f} < 1.0. '
        'Revise augmentation curriculum before proceeding.'
    )

In [ ]:
# ── Append M1 and M2 to results table ────────────────────────────────────────
results_csv = TABLES_DIR / 'val_results_all_models.csv'

new_rows = [
    {'model_id': 'M1', 'description': 'Blind augmentation (no label conditioning)',
     'AP_easy': m1_easy, 'AP_mod': m1_mod, 'AP_hard': m1_hard,
     'aug_p': OPTIMAL_P, 'use_fem': False, 'use_popam': False,
     'fusion_type': 'none', 'use_vis_head': False,
     'seed': m1_cfg.seed, 'epochs': m1_cfg.epochs},
    {'model_id': 'M2', 'description': 'Label-aware augmentation (budget table)',
     'AP_easy': m2_easy, 'AP_mod': m2_mod, 'AP_hard': m2_hard,
     'aug_p': OPTIMAL_P, 'use_fem': False, 'use_popam': False,
     'fusion_type': 'none', 'use_vis_head': False,
     'seed': m2_cfg.seed, 'epochs': m2_cfg.epochs},
]

if results_csv.exists():
    df = pd.read_csv(results_csv)
    df = df[~df['model_id'].isin(['M1', 'M2'])]
    df = pd.concat([df, pd.DataFrame(new_rows)], ignore_index=True)
else:
    df = pd.DataFrame(new_rows)

df.to_csv(results_csv, index=False)
print(f'Saved to: {results_csv}')
df